In [26]:
import os
print(os.getcwd())

D:\DataAnalysis\电商用户行为RFM分析\notebooks


In [27]:
import pandas as pd
df = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')

In [28]:
df.sample(7)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
475677,576910,21494,ROTATING LEAVES T-LIGHT HOLDER,12,2011-11-17 09:51:00,1.25,12477.0,Germany
466358,576328,23581,JUMBO BAG PAISLEY PARK,10,2011-11-14 15:19:00,2.08,13577.0,United Kingdom
374151,569372,22579,WOODEN TREE CHRISTMAS SCANDINAVIAN,12,2011-10-03 16:04:00,0.83,NaN,United Kingdom
388872,570458,21162,TOXIC AREA DOOR HANGER,2,2011-10-10 15:39:00,0.39,14535.0,United Kingdom
369430,569028,21015,DARK BIRD HOUSE TREE DECORATION,24,2011-09-30 10:20:00,0.29,14911.0,EIRE
239412,558049,20984,12 PENCILS TALL TUBE POSY,3,2011-06-24 14:16:00,0.29,15850.0,United Kingdom
392383,570690,23392,SPACEBOY ROCKET LOLLY MAKERS,1,2011-10-11 16:37:00,4.13,NaN,United Kingdom


In [29]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[ns]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [31]:
df.dropna(how='all', inplace=True)   # 全空的行删除

In [32]:
# CustomerID为空 → 无法识别用户，删除
df = df.dropna(subset=['Customer ID'])

# Description（商品描述）为空 → 无法分析商品，删除
df = df.dropna(subset=['Description'])

In [33]:
# 标记是否为退货
df['is_cancel'] = df['Invoice'].astype(str).str.startswith('C')
# 暂时删除退货行做用户分析，但后面单独统计退货
df_clean = df[df['is_cancel'] == False].copy()

In [34]:
df.isna().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
is_cancel      0
dtype: int64

In [35]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate']) #转换日期格式

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 406830 entries, 0 to 541909
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      406830 non-null  object        
 1   StockCode    406830 non-null  object        
 2   Description  406830 non-null  object        
 3   Quantity     406830 non-null  int64         
 4   InvoiceDate  406830 non-null  datetime64[ns]
 5   Price        406830 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      406830 non-null  object        
 8   is_cancel    406830 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 28.3+ MB


In [37]:
df['TotalAmount'] = df['Quantity'] * df['Price'] #添加总金额列，便于后续计算

In [38]:
df.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_cancel,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,False,15.30
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,False,25.50
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,False,11.10
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,False,11.10
9,536368,22960,JAM MAKING SET WITH JARS,6,2010-12-01 08:34:00,4.25,13047.0,United Kingdom,False,25.50


In [39]:
# 看TotalAmount的描述统计
df['TotalAmount'].describe()

# 如果发现极端大的值（比如单笔订单几百万），可以查看
df[df['TotalAmount'] > 10000].head()
# 根据业务常识决定是否删除（电商零售一般单笔不超过几千）

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_cancel,TotalAmount
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,False,77183.6
222671,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098.0,United Kingdom,False,38970.0
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,False,168469.6


In [40]:
#删除极端大的总金额记录
# 保留 TotalAmount <= 10000 的行
df = df[df['TotalAmount'] <= 10000]

In [41]:
# 检验是否还有极端大的值
df[df['TotalAmount'] > 10000].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_cancel,TotalAmount


In [42]:
# 检查清洗后的数据
print("\n清洗完成！数据概览：")
print(df.info())
print(f"\n日期范围: {df['InvoiceDate'].min()} 至 {df['InvoiceDate'].max()}")
print(f"唯一用户数: {df['Customer ID'].nunique()}")
print(f"唯一商品数: {df['StockCode'].nunique()}")


清洗完成！数据概览：
<class 'pandas.core.frame.DataFrame'>
Index: 406827 entries, 0 to 541909
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      406827 non-null  object        
 1   StockCode    406827 non-null  object        
 2   Description  406827 non-null  object        
 3   Quantity     406827 non-null  int64         
 4   InvoiceDate  406827 non-null  datetime64[ns]
 5   Price        406827 non-null  float64       
 6   Customer ID  406827 non-null  float64       
 7   Country      406827 non-null  object        
 8   is_cancel    406827 non-null  bool          
 9   TotalAmount  406827 non-null  float64       
dtypes: bool(1), datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 31.4+ MB
None

日期范围: 2010-12-01 08:26:00 至 2011-12-09 12:50:00
唯一用户数: 4372
唯一商品数: 3684


In [43]:
# 保存清洗后的数据（供后续SQL/Power BI使用）
df.to_csv('../data/cleaned/online_retail_cleaned.csv', index=False)
print("\n已保存为 ../data/cleaned/online_retail_cleaned.csv")


已保存为 ../data/cleaned/online_retail_cleaned.csv


In [44]:
import os
print(os.path.exists('../data/cleaned/online_retail_cleaned.csv'))

True
